In [ ]:
"""
GST Reconciliation Engine
=========================
Compares Books (Inward Register) vs GSTR-2B to find matching invoices.
Follows design.md exactly.

Build order:
  Stage 0  — load, clean, recompute, group books
  Stage 1  — exact matching (1A / 1B / 1C)
  Stage 2  — standardized-invoice matching (same logic, invoice_std)
  Stage 3  — split into blue / nonblue queues
  Stage 4A — AI pass (blue queue)
  Stage 4B — AI pass (nonblue queue)
  Stage 5  — merge + residual + 6-sheet output workbook
"""

from __future__ import annotations

import asyncio
import json
import os
import re
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import networkx as nx
import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()

ModuleNotFoundError: No module named 'networkx'

In [ ]:
# ── config ────────────────────────────────────────────────────────────────────
BOOKS_PATH  = r"C:\Users\subra\OneDrive\Inward Register Feb-25.xlsx"
B2B_PATH    = r"C:\Users\subra\OneDrive\022025_29ADAFS3950J1ZS_GSTR2B_18032025_SSA Firm.xlsx"
OUTPUT_PATH = Path("gst_reconciliation_output.xlsx")

TOL               = 2.0   # ₹2 tolerance
AI_MODEL          = "gpt-4o-mini"
PILE_SIZE_LIMIT   = 15    # max rows per side before exact-invoice pre-pass
BLUE_AUTO_ACCEPT  = 0.90
NONBLUE_AUTO_ACCEPT = 0.95

RATE_COLS = {
    "Taxable @ 0%":  0.00,
    "Taxable @ 5%":  0.05,
    "Taxable @ 12%": 0.12,
    "Taxable @ 18%": 0.18,
    "Taxable @ 28%": 0.28,
}


In [ ]:
def clean_str(val) -> str | None:
    if pd.isna(val):
        return None
    s = str(val).strip().upper()
    return s if s not in ("", "NAN", "<NA>", "NONE", "NAT") else None


def standardize_invoice(val: str | None) -> str | None:
    if not val:
        return None
    s = re.sub(r"[^A-Z0-9]", "", val.upper()).lstrip("0")
    return s or None


def money_match(b: pd.Series, g: pd.Series, tol: float = TOL) -> bool:
    """True when taxable, cgst, sgst, igst AND total_tax are all within tol."""
    return (
        abs(b["taxable"]   - g["taxable"])   <= tol
        and abs(b["cgst"]  - g["cgst"])      <= tol
        and abs(b["sgst"]  - g["sgst"])      <= tol
        and abs(b["igst"]  - g["igst"])      <= tol
        and abs(b["total_tax"] - g["total_tax"]) <= tol
    )



In [ ]:
@dataclass
class Match:
    books_id: str
    b2b_id: str
    stage: str        # "1" | "2"
    sub: str          # "1A" | "1B" | "1C" | "2A" | "2B" | "2C"
    match_reason: str
    taxable_diff: float
    total_tax_diff: float

In [ ]:
def load_books(path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Returns (clean_df, dirty_df). clean_df has canonical money columns."""
    df = pd.read_excel(path, header=1, engine="openpyxl")
    df = df[~(df["Supplier"].isna() & df["Gross Total"].isna())].copy()

    # coerce all numeric inputs
    for col in ["Gross Total", "Taxable Value", "CGST", "SGST", "IGST",
                "Cess", "Exempt", "Others", *RATE_COLS]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    # recompute money from rate columns — never trust cached formula cells
    avail = [c for c in RATE_COLS if c in df.columns]
    is_intra = df["Type"].astype(str).str.upper().str.contains("INTRA", na=False)

    df["taxable"]   = sum(df[c] for c in avail).round(2)
    df["cgst"]      = sum(df[c] * (RATE_COLS[c] / 2) for c in avail).where(is_intra, 0).round(2)
    df["sgst"]      = df["cgst"].copy()
    df["igst"]      = sum(df[c] * RATE_COLS[c] for c in avail).where(~is_intra, 0).round(2)
    df["total_tax"] = (df["cgst"] + df["sgst"] + df["igst"]).round(2)

    cess   = df["Cess"]   if "Cess"   in df.columns else 0
    exempt = df["Exempt"] if "Exempt" in df.columns else 0
    others = df["Others"] if "Others" in df.columns else 0
    df["gross_recomputed"] = (df["taxable"] + df["cgst"] + df["sgst"] + df["igst"]
                               + cess + exempt + others).round(2)
    df["gross_diff"] = (df["Gross Total"] - df["gross_recomputed"]).round(2)

    # clean identity fields
    for col in ["GSTIN/UIN", "Voucher Number", "Supplier"]:
        df[col] = df[col].apply(clean_str)

    dirty = df[df["gross_diff"].abs() > 2].copy()
    clean = df[df["gross_diff"].abs() <= 2].copy()

    print(f"[Stage 0] Books: {len(df)} rows total, dirty={len(dirty)}, clean={len(clean)}")
    return clean, dirty


In [ ]:
def load_2b(path: str) -> pd.DataFrame:
    """Load GSTR-2B B2B sheet with MultiIndex header, return flat canonical frame."""
    df = pd.read_excel(path, sheet_name="B2B", header=[4, 5], engine="openpyxl")

    # drop fully empty rows
    sup_col = ("Trade/Legal name", "Unnamed: 1_level_1")
    val_col = ("Invoice Details", "Invoice Value(₹)")
    df = df[~(df[sup_col].isna() & df[val_col].isna())].copy()

    # flatten MultiIndex: "Level1::Level2" or just "Level1" if level2 is Unnamed
    df.columns = [
        a if "Unnamed" in b else f"{a}::{b}"
        for a, b in df.columns
    ]

    df = df.rename(columns={
        "GSTIN of supplier":                      "gstin",
        "Trade/Legal name":                       "supplier",
        "Invoice Details::Invoice number":        "invoice_raw",
        "Invoice Details::Invoice Date":          "invoice_date",
        "Invoice Details::Invoice Value(₹)": "invoice_value",
        "Taxable Value (₹)":                 "taxable",
        "Tax Amount::Central Tax(₹)":        "cgst",
        "Tax Amount::State/UT Tax(₹)":       "sgst",
        "Tax Amount::Integrated Tax(₹)":     "igst",
    })

    for col in ["taxable", "cgst", "sgst", "igst", "invoice_value"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    df["total_tax"]    = (df["cgst"] + df["sgst"] + df["igst"]).round(2)
    df["invoice_date"] = pd.to_datetime(df.get("invoice_date"), errors="coerce")

    for col in ["gstin", "invoice_raw", "supplier"]:
        df[col] = df[col].apply(clean_str)

    df["invoice_std"] = df["invoice_raw"].apply(standardize_invoice)
    df = df.reset_index(drop=True)
    df.insert(0, "row_id", ["G" + str(i) for i in df.index])

    print(f"[Stage 0] 2B: {len(df)} rows")
    return df

In [ ]:
def group_books(df: pd.DataFrame) -> pd.DataFrame:
    """
    Group books to invoice grain. Grouping key per row (OR, not AND):
      GSTIN present  → (gstin, invoice_raw)
      GSTIN missing  → (supplier, invoice_raw)
      no invoice     → standalone (ungrouped — unsafe to merge with others)
    B2C rows (no GSTIN) are included; grouped by supplier+invoice.
    """
    def make_key(row):
        g, v, s = row["GSTIN/UIN"], row["Voucher Number"], row["Supplier"]
        if g and v:
            return ("gstin_invoice", g, v)
        if not g and v:
            return ("supplier_invoice", s or f"_null_{row.name}", v)
        return ("standalone", f"_solo_{row.name}", f"_solo_{row.name}")

    df = df.copy()
    df[["_ktype", "_k1", "_k2"]] = df.apply(lambda r: pd.Series(make_key(r)), axis=1)

    grouped = (
        df.groupby(["_ktype", "_k1", "_k2"], dropna=False, sort=False)
        .agg(
            gstin        =("GSTIN/UIN",       "first"),
            invoice_raw  =("Voucher Number",  "first"),
            supplier     =("Supplier",        "first"),
            invoice_date =("Accounting Date", "first"),
            taxable      =("taxable",         "sum"),
            cgst         =("cgst",            "sum"),
            sgst         =("sgst",            "sum"),
            igst         =("igst",            "sum"),
            member_row_ids=("Voucher Number", lambda x: list(x.index)),
        )
        .reset_index()
    )
    grouped["total_tax"]    = (grouped["cgst"] + grouped["sgst"] + grouped["igst"]).round(2)
    grouped["invoice_std"]  = grouped["invoice_raw"].apply(standardize_invoice)
    grouped["invoice_date"] = pd.to_datetime(grouped["invoice_date"], errors="coerce")
    grouped = grouped.reset_index(drop=True)
    grouped.insert(0, "row_id", ["B" + str(i) for i in grouped.index])

    n_multi     = (grouped["member_row_ids"].apply(len) > 1).sum()
    n_no_gstin  = grouped["gstin"].isna().sum()
    n_standalone= (grouped["_ktype"] == "standalone").sum()

    print(f"[Stage 0] Grouped books: {len(grouped)} rows | "
          f"multi-line={n_multi} | no-GSTIN={n_no_gstin} | standalone={n_standalone}")
    return grouped


In [ ]:
def _identity(b: pd.Series, g: pd.Series, use_std: bool) -> tuple[bool, bool, bool]:
    """(gstin_match, supplier_match, invoice_match). Nulls never equal."""
    inv_b = b["invoice_std"] if use_std else b["invoice_raw"]
    inv_g = g["invoice_std"] if use_std else g["invoice_raw"]
    gm = bool(b["gstin"]    and g["gstin"]    and b["gstin"]    == g["gstin"])
    sm = bool(b["supplier"] and g["supplier"] and b["supplier"] == g["supplier"])
    im = bool(inv_b         and inv_g         and inv_b         == inv_g)
    return gm, sm, im

In [ ]:
def run_exact_stage(
    books: pd.DataFrame,
    gstr2b: pd.DataFrame,
    use_std: bool = False,
) -> tuple[list[Match], pd.DataFrame, pd.DataFrame]:
    """
    Stage 1 (use_std=False) or Stage 2 (use_std=True).

    Candidate pool = money_match AND (gstin_match OR supplier_match).
    Sub-classification:
      xA — invoice also matches + unique pair → auto-match
      xB — invoice differs + unique loose pair → auto-match (invoice format/voucher note)
      xC — collision cluster → only invoice-exact unique pairs; rest go to Stage 3 blue
    """
    lbl = "2" if use_std else "1"

    # Build all candidate pairs
    candidates: list[dict] = []
    for _, b in books.iterrows():
        for _, g in gstr2b.iterrows():
            if not money_match(b, g):
                continue
            gm, sm, im = _identity(b, g, use_std)
            if not (gm or sm):
                continue
            candidates.append({
                "books_id": b["row_id"],
                "b2b_id":   g["row_id"],
                "gm": gm, "sm": sm, "im": im,
                "taxable_diff":   round(abs(b["taxable"]   - g["taxable"]),   2),
                "total_tax_diff": round(abs(b["total_tax"] - g["total_tax"]), 2),
            })

    used_b: set[str] = set()
    used_g: set[str] = set()
    matches: list[Match] = []

    # ── xA: invoice match + unique ────────────────────────────────────────────
    tight = [c for c in candidates if c["im"]]
    cnt_b = Counter(c["books_id"] for c in tight)
    cnt_g = Counter(c["b2b_id"]   for c in tight)

    for c in tight:
        if (cnt_b[c["books_id"]] == 1 and cnt_g[c["b2b_id"]] == 1
                and c["books_id"] not in used_b and c["b2b_id"] not in used_g):
            matches.append(Match(
                c["books_id"], c["b2b_id"], lbl, f"{lbl}A",
                "exact identity + invoice + money",
                c["taxable_diff"], c["total_tax_diff"],
            ))
            used_b.add(c["books_id"])
            used_g.add(c["b2b_id"])

    # ── xC: collision clusters — take invoice-exact unique pairs inside cluster
    collision_b = {bid for bid, n in cnt_b.items() if n > 1 and bid not in used_b}
    collision_g = {gid for gid, n in cnt_g.items() if n > 1 and gid not in used_g}
    if collision_b or collision_g:
        ct2 = [c for c in tight if c["books_id"] in collision_b or c["b2b_id"] in collision_g]
        cc_b = Counter(c["books_id"] for c in ct2)
        cc_g = Counter(c["b2b_id"]   for c in ct2)
        for c in ct2:
            if (cc_b[c["books_id"]] == 1 and cc_g[c["b2b_id"]] == 1
                    and c["books_id"] not in used_b and c["b2b_id"] not in used_g):
                matches.append(Match(
                    c["books_id"], c["b2b_id"], lbl, f"{lbl}C",
                    "exact match within collision cluster",
                    c["taxable_diff"], c["total_tax_diff"],
                ))
                used_b.add(c["books_id"])
                used_g.add(c["b2b_id"])

    # ── xB: invoice differs, unique loose pair ────────────────────────────────
    loose = [c for c in candidates
             if not c["im"] and c["books_id"] not in used_b and c["b2b_id"] not in used_g]
    lb_cnt = Counter(c["books_id"] for c in loose)
    lg_cnt = Counter(c["b2b_id"]   for c in loose)
    for c in loose:
        if (lb_cnt[c["books_id"]] == 1 and lg_cnt[c["b2b_id"]] == 1
                and c["books_id"] not in used_b and c["b2b_id"] not in used_g):
            matches.append(Match(
                c["books_id"], c["b2b_id"], lbl, f"{lbl}B",
                "identity + money match; invoice differs (format/voucher)",
                c["taxable_diff"], c["total_tax_diff"],
            ))
            used_b.add(c["books_id"])
            used_g.add(c["b2b_id"])

    rem_books = books[~books["row_id"].isin(used_b)].copy()
    rem_2b    = gstr2b[~gstr2b["row_id"].isin(used_g)].copy()

    print(f"[Stage {lbl}] matches={len(matches)} | "
          f"remaining books={len(rem_books)}, 2B={len(rem_2b)}")
    return matches, rem_books, rem_2b


In [ ]:

def run_stage3(
    books: pd.DataFrame,
    gstr2b: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    BLUE    = money + (gstin OR supplier) match, but invoice mismatch survived Stages 1-2.
    NONBLUE = money coincidence only (no identity anchor).
    Books with no money-twin at all → immediately books-only unmatched (skip AI).

    Returns: (blue_books, blue_2b, nonblue_books, no_money_twin_books)
    """
    blue_b: set[str]        = set()
    nonblue_b: set[str]     = set()
    no_twin_b: set[str]     = set()
    blue_g: set[str]        = set()

    for _, b in books.iterrows():
        twins = [g for _, g in gstr2b.iterrows() if money_match(b, g)]
        if not twins:
            no_twin_b.add(b["row_id"])
            continue
        id_twins = [g for g in twins
                    if (b["gstin"]    and g["gstin"]    and b["gstin"]    == g["gstin"])
                    or (b["supplier"] and g["supplier"] and b["supplier"] == g["supplier"])]
        if id_twins:
            blue_b.add(b["row_id"])
            for g in id_twins:
                blue_g.add(g["row_id"])
        else:
            nonblue_b.add(b["row_id"])

    q_blue_b    = books[books["row_id"].isin(blue_b)].copy()
    q_blue_g    = gstr2b[gstr2b["row_id"].isin(blue_g)].copy()
    q_nonblue_b = books[books["row_id"].isin(nonblue_b)].copy()
    q_no_twin   = books[books["row_id"].isin(no_twin_b)].copy()

    print(f"[Stage 3] blue_books={len(q_blue_b)} | nonblue_books={len(q_nonblue_b)} | "
          f"books-only (no money twin)={len(q_no_twin)}")
    return q_blue_b, q_blue_g, q_nonblue_b, q_no_twin


In [ ]:
def build_piles(books: pd.DataFrame, gstr2b: pd.DataFrame) -> list[dict]:
    """
    Partition rows into disjoint piles via connected components.
    Each pile = {"books": [Series, ...], "2b": [Series, ...]}
    Applies exact-invoice pre-pass for oversized piles.
    """
    G = nx.Graph()
    for _, b in books.iterrows():
        G.add_node(("B", b["row_id"]), data=b.to_dict())
    for _, g in gstr2b.iterrows():
        G.add_node(("G", g["row_id"]), data=g.to_dict())
    for _, b in books.iterrows():
        for _, g in gstr2b.iterrows():
            if money_match(b, g):
                G.add_edge(("B", b["row_id"]), ("G", g["row_id"]))

    piles: list[dict] = []
    for comp in nx.connected_components(G):
        b_rows = [G.nodes[n]["data"] for n in comp if n[0] == "B"]
        g_rows = [G.nodes[n]["data"] for n in comp if n[0] == "G"]

        if not b_rows or not g_rows:
            piles.append({"books": b_rows, "2b": g_rows})
            continue

        if len(b_rows) > PILE_SIZE_LIMIT or len(g_rows) > PILE_SIZE_LIMIT:
            # exact-invoice pre-pass: pair rows with matching invoice_std
            paired_b: set[str] = set()
            paired_g: set[str] = set()
            for b in b_rows:
                for g in g_rows:
                    bs = pd.Series(b)
                    gs = pd.Series(g)
                    if (b["invoice_std"] and g["invoice_std"]
                            and b["invoice_std"] == g["invoice_std"]
                            and money_match(bs, gs)
                            and b["row_id"] not in paired_b
                            and g["row_id"] not in paired_g):
                        paired_b.add(b["row_id"])
                        paired_g.add(g["row_id"])
            b_rem = [r for r in b_rows if r["row_id"] not in paired_b]
            g_rem = [r for r in g_rows if r["row_id"] not in paired_g]
            if b_rem or g_rem:
                piles.append({"books": b_rem, "2b": g_rem})
        else:
            piles.append({"books": b_rows, "2b": g_rows})

    return piles

In [ ]:

_TOOL = {
    "type": "function",
    "function": {
        "name": "submit_pile_matches",
        "description": "Submit matching decisions. Every row ID must appear exactly once.",
        "parameters": {
            "type": "object",
            "required": ["reasoning", "matches", "unmatched_books", "unmatched_b2b"],
            "properties": {
                "reasoning": {"type": "string"},
                "matches": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "required": ["books_id", "b2b_id", "reason", "confidence"],
                        "properties": {
                            "books_id":   {"type": "string"},
                            "b2b_id":     {"type": "string"},
                            "reason":     {"type": "string"},
                            "confidence": {"type": "number"},
                        },
                    },
                },
                "unmatched_books": {
                    "type": "array",
                    "items": {"type": "object", "required": ["books_id", "reason"],
                              "properties": {"books_id": {"type": "string"},
                                             "reason": {"type": "string"}}},
                },
                "unmatched_b2b": {
                    "type": "array",
                    "items": {"type": "object", "required": ["b2b_id", "reason"],
                              "properties": {"b2b_id": {"type": "string"},
                                             "reason": {"type": "string"}}},
                },
            },
        },
    },
}


def _pile_prompt(pile: dict, pass_type: str) -> str:
    question = (
        "Are these the same invoice despite a different invoice number?"
        if pass_type == "blue"
        else "Are these the same party/transaction at all? (low auto-accept — treat as suggestion)"
    )
    lines = [
        f"GST reconciliation — {pass_type} AI pass.",
        "All rows in this pile already agree on money within ₹2. Decide identity only.",
        "One-to-one constraint: each books row matches at most one 2B row and vice versa.",
        "Every row ID must appear exactly once across matches / unmatched_books / unmatched_b2b.",
        "",
        "Columns: gstin = supplier tax ID | invoice = invoice number | "
        "supplier = trade name | taxable/cgst/sgst/igst = money fields.",
        "",
        f"Question: {question}",
        "",
        "BOOKS rows:",
    ]
    for r in pile["books"]:
        lines.append(
            f"  {r['row_id']}: gstin={r['gstin']} invoice={r['invoice_raw']} "
            f"supplier={r['supplier']} taxable={r['taxable']} "
            f"cgst={r['cgst']} sgst={r['sgst']} igst={r['igst']}"
        )
    lines += ["", "2B rows:"]
    for r in pile["2b"]:
        lines.append(
            f"  {r['row_id']}: gstin={r['gstin']} invoice={r['invoice_raw']} "
            f"supplier={r['supplier']} taxable={r['taxable']} "
            f"cgst={r['cgst']} sgst={r['sgst']} igst={r['igst']}"
        )
    lines += ["", "Call submit_pile_matches. Every Bi and every Gj must appear exactly once."]
    return "\n".join(lines)


async def _ai_pile(client: AsyncOpenAI, pile: dict, pass_type: str) -> dict | None:
    if not pile["books"] or not pile["2b"]:
        return None

    response = await client.chat.completions.create(
        model=AI_MODEL,
        temperature=0,
        tools=[_TOOL],
        tool_choice={"type": "function", "function": {"name": "submit_pile_matches"}},
        messages=[{"role": "user", "content": _pile_prompt(pile, pass_type)}],
    )

    raw = json.loads(response.choices[0].message.tool_calls[0].function.arguments)

    # validate + re-check money_match for every proposed pair
    valid_b = {r["row_id"]: r for r in pile["books"]}
    valid_g = {r["row_id"]: r for r in pile["2b"]}
    seen_b: set[str] = set()
    seen_g: set[str] = set()
    clean_matches = []

    for m in raw.get("matches", []):
        bid, gid = m["books_id"], m["b2b_id"]
        if bid not in valid_b or gid not in valid_g:
            continue
        if bid in seen_b or gid in seen_g:
            continue
        if not money_match(pd.Series(valid_b[bid]), pd.Series(valid_g[gid])):
            continue
        seen_b.add(bid)
        seen_g.add(gid)
        m["_pass"] = pass_type
        clean_matches.append(m)

    raw["matches"] = clean_matches
    raw["_pass"]   = pass_type
    return raw


async def run_ai_stages(
    q_blue_b: pd.DataFrame,
    q_blue_g: pd.DataFrame,
    q_nonblue_b: pd.DataFrame,
    gstr2b_nonblue: pd.DataFrame,
) -> list[dict]:
    """Stage 4A (blue) + 4B (nonblue). Returns validated AI match proposals."""
    client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    piles_blue    = build_piles(q_blue_b,    q_blue_g)
    piles_nonblue = build_piles(q_nonblue_b, gstr2b_nonblue)

    tasks = (
        [_ai_pile(client, p, "blue")    for p in piles_blue    if p["books"] and p["2b"]]
        + [_ai_pile(client, p, "nonblue") for p in piles_nonblue if p["books"] and p["2b"]]
    )

    results = await asyncio.gather(*tasks, return_exceptions=True)
    proposals: list[dict] = []
    for r in results:
        if isinstance(r, Exception):
            print(f"  [AI error] {r}")
        elif r:
            for m in r.get("matches", []):
                m.setdefault("_pass", r["_pass"])
                proposals.append(m)

    print(f"[Stage 4A/4B] AI proposals: {len(proposals)}")
    return proposals


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 5 — GREEDY MERGE + 6-SHEET OUTPUT WORKBOOK
# ═══════════════════════════════════════════════════════════════════════════════

def run_stage5(
    det_matches: list[Match],
    ai_proposals: list[dict],
    books_grouped: pd.DataFrame,
    gstr2b: pd.DataFrame,
    q_no_twin: pd.DataFrame,
    dirty_books: pd.DataFrame,
    output_path: Path,
) -> None:
    books_idx = books_grouped.set_index("row_id")
    b2b_idx   = gstr2b.set_index("row_id")

    used_b: set[str] = {m.books_id for m in det_matches}
    used_g: set[str] = {m.b2b_id   for m in det_matches}

    # greedy accept AI proposals (sorted by confidence desc)
    ai_proposals_sorted = sorted(ai_proposals, key=lambda x: x.get("confidence", 0), reverse=True)
    ai_accepted: list[dict] = []
    ai_review:   list[dict] = []

    for p in ai_proposals_sorted:
        bid, gid = p["books_id"], p["b2b_id"]
        if bid in used_b or gid in used_g:
            continue
        threshold = BLUE_AUTO_ACCEPT if p.get("_pass") == "blue" else NONBLUE_AUTO_ACCEPT
        if p.get("confidence", 0) >= threshold:
            ai_accepted.append(p)
            used_b.add(bid)
            used_g.add(gid)
        else:
            ai_review.append(p)

    # ── matched sheet ─────────────────────────────────────────────────────────
    matched_rows = []
    for m in det_matches:
        b = books_idx.loc[m.books_id]
        g = b2b_idx.loc[m.b2b_id]
        matched_rows.append({
            "match_id":       f"{m.books_id}_{m.b2b_id}",
            "stage":          m.stage,
            "sub":            m.sub,
            "books_row_ids":  str(b.get("member_row_ids", [])),
            "b2b_row_id":     m.b2b_id,
            "gstin":          b.get("gstin"),
            "invoice_books":  b.get("invoice_raw"),
            "invoice_2b":     g.get("invoice_raw"),
            "supplier_books": b.get("supplier"),
            "supplier_2b":    g.get("supplier"),
            "taxable_diff":   m.taxable_diff,
            "total_tax_diff": m.total_tax_diff,
            "match_reason":   m.match_reason,
            "ai_confidence":  "",
            "ai_reason":      "",
            "flag":           "none",
        })

    for p in ai_accepted:
        b = books_idx.loc[p["books_id"]]
        g = b2b_idx.loc[p["b2b_id"]]
        matched_rows.append({
            "match_id":       f"{p['books_id']}_{p['b2b_id']}",
            "stage":          "4",
            "sub":            "4A" if p.get("_pass") == "blue" else "4B",
            "books_row_ids":  str(b.get("member_row_ids", [])),
            "b2b_row_id":     p["b2b_id"],
            "gstin":          b.get("gstin"),
            "invoice_books":  b.get("invoice_raw"),
            "invoice_2b":     g.get("invoice_raw"),
            "supplier_books": b.get("supplier"),
            "supplier_2b":    g.get("supplier"),
            "taxable_diff":   round(abs(b["taxable"] - g["taxable"]), 2),
            "total_tax_diff": round(abs(b["total_tax"] - g["total_tax"]), 2),
            "match_reason":   p.get("reason", ""),
            "ai_confidence":  p.get("confidence", ""),
            "ai_reason":      p.get("reason", ""),
            "flag":           "blue" if p.get("_pass") == "blue" else "none",
        })

    df_matched = pd.DataFrame(matched_rows)

    # ── remaining sheets ──────────────────────────────────────────────────────
    unmatched_b_ids = set(books_grouped["row_id"]) - used_b
    df_unmatched_b  = pd.concat([
        books_grouped[books_grouped["row_id"].isin(unmatched_b_ids)],
        q_no_twin,
    ], ignore_index=True)

    unmatched_g_ids = set(gstr2b["row_id"]) - used_g
    df_unmatched_g  = gstr2b[gstr2b["row_id"].isin(unmatched_g_ids)].copy()

    df_blue_review = pd.DataFrame(ai_review) if ai_review else pd.DataFrame()

    # no_gstin_b2c: books rows where gstin is null (tracked for output)
    df_b2c = books_grouped[books_grouped["gstin"].isna()].copy()

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df_matched.to_excel(    writer, sheet_name="matched",          index=False)
        df_blue_review.to_excel(writer, sheet_name="blue_review",      index=False)
        df_unmatched_b.to_excel(writer, sheet_name="unmatched_books",  index=False)
        df_unmatched_g.to_excel(writer, sheet_name="unmatched_2b",     index=False)
        df_b2c.to_excel(        writer, sheet_name="no_gstin_b2c",     index=False)
        dirty_books.to_excel(   writer, sheet_name="dirty_input",      index=False)

    print(f"\n[Stage 5] Output → {output_path}")
    print(f"  matched:         {len(df_matched)}")
    print(f"  blue_review:     {len(df_blue_review)}")
    print(f"  unmatched_books: {len(df_unmatched_b)}")
    print(f"  unmatched_2b:    {len(df_unmatched_g)}")
    print(f"  no_gstin_b2c:    {len(df_b2c)}")
    print(f"  dirty_input:     {len(dirty_books)}")



In [ ]:

async def main() -> None:
    # Stage 0
    print("\n── Stage 0: Load & Prepare ──")
    books_clean, books_dirty = load_books(BOOKS_PATH)
    gstr2b = load_2b(B2B_PATH)
    books_grouped = group_books(books_clean)

    # Stage 1
    print("\n── Stage 1: Exact Matching ──")
    m1, books_r1, gstr2b_r1 = run_exact_stage(books_grouped, gstr2b, use_std=False)

    # Stage 2
    print("\n── Stage 2: Standardized Invoice ──")
    m2, books_r2, gstr2b_r2 = run_exact_stage(books_r1, gstr2b_r1, use_std=True)

    # Stage 3
    print("\n── Stage 3: Blue / Nonblue Split ──")
    q_blue_b, q_blue_g, q_nonblue_b, q_no_twin = run_stage3(books_r2, gstr2b_r2)

    used_in_blue   = set(q_blue_g["row_id"]) if len(q_blue_g) else set()
    gstr2b_nonblue = gstr2b_r2[~gstr2b_r2["row_id"].isin(used_in_blue)].copy()

    # Stage 4A / 4B
    print("\n── Stage 4A/4B: AI Matching ──")
    ai_proposals = await run_ai_stages(q_blue_b, q_blue_g, q_nonblue_b, gstr2b_nonblue)

    # Stage 5
    print("\n── Stage 5: Merge & Output ──")
    run_stage5(
        det_matches=m1 + m2,
        ai_proposals=ai_proposals,
        books_grouped=books_grouped,
        gstr2b=gstr2b,
        q_no_twin=q_no_twin,
        dirty_books=books_dirty,
        output_path=OUTPUT_PATH,
    )


if __name__ == "__main__":
    asyncio.run(main())
